# Module 3 • Classical Natural Language Processing

# Lesson 11 • Text Preprocessing and Normalization

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner  
**Estimated study time:** 90–120 minutes

---

## Scope

This lesson explains how raw text is inspected, validated, cleaned, and
normalized before feature extraction or modeling. It emphasizes that
preprocessing must preserve the information required by the task.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish cleaning, normalization, tokenization, stemming, and
  lemmatization;
- inspect raw text before transforming it;
- handle Unicode and text encoding correctly;
- normalize whitespace, HTML entities, URLs, email addresses, and user
  mentions;
- make task-aware decisions about case, punctuation, numbers, emojis, and
  repeated characters;
- explain why stop-word removal may help or harm a task;
- preserve negation and other meaningful linguistic signals;
- apply configurable English and Arabic normalization;
- build and test a reproducible preprocessing function;
- evaluate preprocessing by its downstream effect rather than visual
  cleanliness alone.

## Table of Contents

1. What Is Text Preprocessing?
2. Preprocessing Is Task-Dependent
3. Inspect Before Cleaning
4. Unicode and Encoding
5. Whitespace and Control Characters
6. HTML and Escaped Text
7. URLs, Emails, Mentions, and Identifiers
8. Case Normalization
9. Punctuation and Symbols
10. Numbers, Dates, and Currency
11. Emojis, Emoticons, and Repeated Characters
12. Contractions and Spelling Variation
13. Stop Words and Negation
14. Arabic Text Normalization
15. Configurable Preprocessing Pipeline
16. Testing and Reproducibility
17. Evaluating Preprocessing Decisions
18. Knowledge Check
19. Exercises
20. Summary and Next Lesson

# 1. What Is Text Preprocessing?

**Text preprocessing** is the set of operations used to make raw language data
suitable for analysis or modeling.

It may include:

- input validation;
- decoding and Unicode normalization;
- whitespace normalization;
- markup removal;
- masking structured identifiers;
- case handling;
- punctuation and symbol handling;
- language-specific normalization;
- tokenization;
- optional stemming or lemmatization.

These operations are related but not interchangeable.

## 1.1 Important Distinctions

| Operation | Purpose |
|---|---|
| Cleaning | remove or repair unwanted artifacts |
| Normalization | map selected variants to a consistent form |
| Tokenization | divide text into processing units |
| Stemming | heuristically reduce word forms |
| Lemmatization | map forms to dictionary lemmas |
| Feature extraction | convert text into numerical representations |

This lesson focuses on cleaning and normalization. Tokenization is studied in
the next lesson.

> **Key Idea**
>
> The best preprocessing pipeline is not the one that changes the most text.
> It is the one that removes irrelevant variation while preserving task-relevant
> evidence.

# 2. Preprocessing Is Task-Dependent

The same transformation can help one task and damage another.

| Transformation | May help | May harm |
|---|---|---|
| Lowercasing | topic classification | Named Entity Recognition |
| Punctuation removal | some count-based models | sentiment, dialogue, sentence boundaries |
| Number replacement | general topic modeling | finance, dates, dosage extraction |
| Emoji removal | formal-document analysis | sentiment and social-media analysis |
| Stop-word removal | some retrieval pipelines | negation, authorship, generation |
| Diacritic removal | Arabic search recall | diacritization and lexical disambiguation |

In [ ]:
import pandas as pd

task_decisions = pd.DataFrame(
    [
        ("Sentiment analysis", "keep negation, punctuation, emojis"),
        ("Named Entity Recognition", "preserve case and boundaries"),
        ("Topic classification", "case normalization may be acceptable"),
        ("Financial extraction", "preserve numbers, dates, and currency"),
        ("Arabic search", "document normalization and diacritic policy"),
    ],
    columns=["Task", "Important preprocessing decision"],
)

task_decisions

# 3. Inspect Before Cleaning

Before writing normalization rules, inspect:

- representative samples;
- missing and empty values;
- data types;
- languages and scripts;
- message length;
- HTML and markup;
- URLs and identifiers;
- punctuation and emoji;
- spelling variation;
- duplicates;
- unusual control characters.

Cleaning rules based on a few convenient examples may fail on the actual
dataset.

In [ ]:
raw_examples = pd.DataFrame(
    [
        (1, "  GREAT service!!! 😊  "),
        (2, "<p>Reset your password&nbsp;here.</p>"),
        (3, "Email me at sara@example.com"),
        (4, "Visit https://example.com/help?id=45"),
        (5, "The price is $1,250.50"),
        (6, "لا أستطيع تسجيل الدخول!!!"),
        (7, None),
        (8, "Line one\n\nLine two\twith tabs"),
    ],
    columns=["id", "text"],
)

raw_examples

In [ ]:
print("Missing values:")
print(raw_examples.isna().sum())

raw_examples["character_count"] = (
    raw_examples["text"].fillna("").str.len()
)

raw_examples[["id", "character_count"]]

# 4. Unicode and Encoding

Python 3 strings are Unicode. Text files still require an encoding when read
from or written to bytes.

UTF-8 is a common default:

```python
Path("file.txt").read_text(encoding="utf-8")
```

Incorrect decoding may produce errors or corrupted characters.

In [ ]:
multilingual_text = [
    "Natural language processing",
    "معالجة اللغة الطبيعية",
    "traitement automatique du langage",
    "自然语言处理",
    "NLP 😊",
]

for text in multilingual_text:
    print(
        f"{text!r:<42} "
        f"characters={len(text):>2} "
        f"utf8_bytes={len(text.encode('utf-8')):>2}"
    )

## 4.1 Unicode Normalization

Visually identical text may contain different code-point sequences.

Common normalization forms include:

- NFC: canonical composition;
- NFD: canonical decomposition;
- NFKC: compatibility composition;
- NFKD: compatibility decomposition.

Compatibility normalization may change formatting distinctions, so it should
be selected deliberately.

In [ ]:
import unicodedata

composed = "café"
decomposed = "cafe\u0301"

print("Equal before normalization:", composed == decomposed)
print("Composed:", [hex(ord(ch)) for ch in composed])
print("Decomposed:", [hex(ord(ch)) for ch in decomposed])

normalized_a = unicodedata.normalize("NFC", composed)
normalized_b = unicodedata.normalize("NFC", decomposed)

print("Equal after NFC:", normalized_a == normalized_b)

# 5. Whitespace and Control Characters

Raw data may contain:

- leading and trailing spaces;
- repeated spaces;
- tabs;
- newlines;
- non-breaking spaces;
- invisible control characters.

Collapsing whitespace is often safe for sentence-level classification, but
line breaks may carry structure in poetry, source code, addresses, tables, or
document segmentation.

In [ ]:
import re

WHITESPACE_PATTERN = re.compile(r"\s+")

def normalize_whitespace(text: str) -> str:
    return WHITESPACE_PATTERN.sub(" ", text).strip()


whitespace_example = "  Line one\n\nLine two\twith   spaces  "

print("Before:", repr(whitespace_example))
print("After: ", repr(normalize_whitespace(whitespace_example)))

# 6. HTML and Escaped Text

Web-derived text may contain HTML tags and entities:

```text
<p>Natural&nbsp;Language <strong>Processing</strong></p>
```

Python's standard library can decode entities. Simple tag removal can support
controlled examples, but complex HTML should be parsed with a dedicated HTML
parser.

In [ ]:
import html

TAG_PATTERN = re.compile(r"<[^>]+>")

def simple_html_to_text(value: str) -> str:
    without_tags = TAG_PATTERN.sub(" ", value)
    decoded = html.unescape(without_tags)
    return normalize_whitespace(decoded)


html_example = (
    "<p>Natural&nbsp;Language "
    "<strong>Processing</strong></p>"
)

print(simple_html_to_text(html_example))

A regex cannot reliably parse every valid or malformed HTML document. Use this
simplified approach only when the input format is controlled.

# 7. URLs, Emails, Mentions, and Identifiers

Structured strings can be:

- removed;
- retained;
- replaced with placeholders;
- extracted into separate features.

Replacing a value with a placeholder often preserves the fact that a type of
item occurred without retaining the exact value.

In [ ]:
URL_PATTERN = re.compile(
    r"https?://\S+|www\.\S+",
    flags=re.IGNORECASE,
)
EMAIL_PATTERN = re.compile(
    r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b"
)
MENTION_PATTERN = re.compile(r"(?<!\w)@\w+")
ORDER_PATTERN = re.compile(
    r"\b(?:order|ticket)[-_ ]?\d+\b",
    flags=re.IGNORECASE,
)

structured_example = (
    "Hi @support, email sara@example.com or visit "
    "https://example.com. Ticket-1045 is still open."
)

masked = EMAIL_PATTERN.sub(" EMAIL_ADDRESS ", structured_example)
masked = URL_PATTERN.sub(" URL ", masked)
masked = MENTION_PATTERN.sub(" USER_MENTION ", masked)
masked = ORDER_PATTERN.sub(" RECORD_ID ", masked)
masked = normalize_whitespace(masked)

print(masked)

Placeholder tokens should be consistent. A model may treat `URL`, `<URL>`,
and `url_placeholder` as different features.

# 8. Case Normalization

Lowercasing reduces variation:

```text
Model, model, MODEL → model
```

It may also remove useful information:

```text
US  → country abbreviation
us  → pronoun
Apple → organization
apple → fruit
```

In [ ]:
case_examples = [
    "US",
    "us",
    "Apple",
    "apple",
    "WHO",
    "who",
]

for item in case_examples:
    print(f"{item:<8} -> {item.lower()}")

**Case folding** is a stronger Unicode-aware normalization than simple
lowercasing and is useful for caseless comparison.

In [ ]:
casefold_examples = ["Straße", "STRASSE", "Μάϊος"]

for item in casefold_examples:
    print(f"{item!r:<12} lower={item.lower()!r:<12} casefold={item.casefold()!r}")

# 9. Punctuation and Symbols

Punctuation may communicate:

- sentence boundaries;
- questions;
- emphasis;
- lists;
- quotations;
- abbreviations;
- emotion;
- code or mathematical structure.

Compare:

```text
Let's eat, Grandma!
Let's eat Grandma!
```

Blind punctuation removal can alter meaning.

In [ ]:
punctuation_examples = pd.DataFrame(
    [
        ("Really?", "question or doubt"),
        ("Really!", "emphasis or surprise"),
        ("Really...", "hesitation or implication"),
        ("not good", "negative evaluation"),
        ("not, good", "different or malformed structure"),
    ],
    columns=["Text", "Possible signal"],
)

punctuation_examples

In [ ]:
PUNCTUATION_PATTERN = re.compile(r"[^\w\s]", flags=re.UNICODE)

sentence = "This is useful—really useful!"

removed = PUNCTUATION_PATTERN.sub(" ", sentence)
removed = normalize_whitespace(removed)

print("Original:", sentence)
print("Removed: ", removed)

Use punctuation removal only when its lost information is irrelevant to the
task.

# 10. Numbers, Dates, and Currency

Numbers may represent:

- quantities;
- dates;
- times;
- prices;
- measurements;
- versions;
- identifiers;
- rankings.

Replacing every number with `NUMBER` may reduce sparsity, but it also merges
semantically different values.

In [ ]:
NUMBER_PATTERN = re.compile(r"\b\d+(?:[.,]\d+)*\b")
ISO_DATE_PATTERN = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
CURRENCY_PATTERN = re.compile(
    r"(?:[$€£]\s?\d+(?:,\d{3})*(?:\.\d+)?|\b\d+(?:\.\d+)?\s?(?:USD|EUR|GBP)\b)",
    flags=re.IGNORECASE,
)

number_text = (
    "Version 3.2 costs $1,250.50 and was released on 2026-07-25."
)

print("Dates:", ISO_DATE_PATTERN.findall(number_text))
print("Currency:", CURRENCY_PATTERN.findall(number_text))
print("Numbers:", NUMBER_PATTERN.findall(number_text))

Specific patterns such as dates and currency should usually be processed
before a broad number pattern.

# 11. Emojis, Emoticons, and Repeated Characters

Emojis and emoticons may signal:

- sentiment;
- irony;
- politeness;
- topic;
- reaction;
- community-specific meaning.

Repeated punctuation or characters may express intensity:

```text
good
goood
goooood!!!
```

In [ ]:
expressive_examples = pd.DataFrame(
    [
        ("Great 😊", "positive affect"),
        ("Great 🙄", "possible irony or annoyance"),
        ("Nooooo!", "intensified rejection"),
        ("Really???", "strong questioning or disbelief"),
    ],
    columns=["Text", "Possible interpretation"],
)

expressive_examples

In [ ]:
REPEATED_CHARACTER_PATTERN = re.compile(r"(.)\1{2,}")

def limit_repeated_characters(text: str, maximum: int = 2) -> str:
    return REPEATED_CHARACTER_PATTERN.sub(
        lambda match: match.group(1) * maximum,
        text,
    )


for text in ["soooo good!!!", "nooooo", "cool"]:
    print(f"{text:<15} -> {limit_repeated_characters(text)}")

Character compression is language- and task-dependent. Repetition may be
noise, emphasis, or a valid spelling pattern.

# 12. Contractions and Spelling Variation

English contractions include:

```text
don't, isn't, we're, I'd
```

Expansion may help some rule-based systems:

```text
don't → do not
```

However, apostrophes can also mark possession, quotations, or names.

In [ ]:
contraction_map = {
    "can't": "can not",
    "cannot": "can not",
    "don't": "do not",
    "isn't": "is not",
    "won't": "will not",
    "we're": "we are",
}

def expand_known_contractions(text: str) -> str:
    words = text.split()
    expanded = [
        contraction_map.get(word.lower(), word)
        for word in words
    ]
    return " ".join(expanded)


print(expand_known_contractions("we're ready but don't start"))

Spelling correction should also be used carefully. A rare term, name, product,
dialect form, or technical identifier may be incorrectly changed into a common
dictionary word.

# 13. Stop Words and Negation

**Stop words** are high-frequency words sometimes removed from sparse
representations.

Examples may include:

```text
the, is, of, and
```

There is no universal stop-word list.

In [ ]:
stop_words = {
    "the", "is", "of", "and", "a", "an", "to",
}

sentence = "the model is not accurate"

filtered = [
    token for token in sentence.split()
    if token not in stop_words
]

print("Original:", sentence.split())
print("Filtered:", filtered)

The word `not` was deliberately preserved. Removing negation can reverse
meaning:

```text
useful
not useful
```

Stop-word removal may also harm:

- authorship analysis;
- grammar-sensitive tasks;
- phrase retrieval;
- question answering;
- translation;
- generation;
- Transformer models trained on natural text.

# 14. Arabic Text Normalization

Arabic preprocessing may consider:

- Unicode normalization;
- Tatweel removal;
- optional diacritic removal;
- normalization of selected Alef forms;
- normalization of Alef Maqsura and Ya;
- Teh Marbuta policy;
- Arabic and Persian digit variants;
- punctuation and whitespace;
- dialect spelling variation;
- Arabizi and code-switching.

Each decision must match the language variety and task.

In [ ]:
ARABIC_DIACRITICS_PATTERN = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)
TATWEEL_PATTERN = re.compile("\u0640")

ARABIC_CHARACTER_MAP = str.maketrans(
    {
        "أ": "ا",
        "إ": "ا",
        "آ": "ا",
        "ى": "ي",
    }
)

def normalize_arabic(
    text: str,
    remove_diacritics: bool = True,
    remove_tatweel: bool = True,
    normalize_selected_letters: bool = True,
) -> str:
    result = unicodedata.normalize("NFC", text)

    if remove_tatweel:
        result = TATWEEL_PATTERN.sub("", result)

    if remove_diacritics:
        result = ARABIC_DIACRITICS_PATTERN.sub("", result)

    if normalize_selected_letters:
        result = result.translate(ARABIC_CHARACTER_MAP)

    return normalize_whitespace(result)


arabic_samples = [
    "مُعَالَجَةُ اللُّغَةِ الطَّبِيعِيَّةِ",
    "الـــلغة العربية",
    "إلى المدرسة",
    "على الطريق",
]

for sample in arabic_samples:
    print(f"{sample} -> {normalize_arabic(sample)}")

## 14.1 Information Loss in Arabic Normalization

Normalizing:

```text
أ، إ، آ → ا
ى → ي
```

may improve matching recall but merge distinct orthographic forms. Removing
diacritics may also increase ambiguity.

A diacritization, morphology, authorship, or language-learning task may require
preserving distinctions that a search system chooses to normalize.

# 15. Configurable Preprocessing Pipeline

A reusable preprocessing function should make transformations explicit and
configurable.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class NormalizationConfig:
    unicode_form: str = "NFKC"
    decode_html_entities: bool = True
    remove_html_tags: bool = True
    mask_emails: bool = True
    mask_urls: bool = True
    mask_mentions: bool = True
    lowercase: bool = True
    normalize_spaces: bool = True
    limit_character_repetition: bool = False


def preprocess_text(
    value: object,
    config: NormalizationConfig = NormalizationConfig(),
) -> str:
    if value is None:
        return ""

    if not isinstance(value, str):
        value = str(value)

    text = unicodedata.normalize(config.unicode_form, value)

    if config.decode_html_entities:
        text = html.unescape(text)

    if config.remove_html_tags:
        text = TAG_PATTERN.sub(" ", text)

    if config.mask_emails:
        text = EMAIL_PATTERN.sub(" EMAIL_ADDRESS ", text)

    if config.mask_urls:
        text = URL_PATTERN.sub(" URL ", text)

    if config.mask_mentions:
        text = MENTION_PATTERN.sub(" USER_MENTION ", text)

    if config.limit_character_repetition:
        text = limit_repeated_characters(text)

    if config.lowercase:
        text = text.lower()

    if config.normalize_spaces:
        text = normalize_whitespace(text)

    return text

In [ ]:
examples = [
    "  GREAT service!!! 😊  ",
    "<p>Email SARA@example.com</p>",
    "Visit https://example.com @support",
    None,
]

default_config = NormalizationConfig()

for example in examples:
    print("Before:", repr(example))
    print("After: ", repr(preprocess_text(example, default_config)))
    print()

The default pipeline deliberately keeps punctuation, numbers, and emojis. They
should be changed only when the task requires it.

## 15.1 Applying the Pipeline to a DataFrame

In [ ]:
working_data = raw_examples.copy()

working_data["normalized_text"] = (
    working_data["text"]
    .map(preprocess_text)
)

working_data[["id", "text", "normalized_text"]]

Store raw text and normalized text in separate columns. Overwriting the source
removes the ability to audit or revise preprocessing decisions.

# 16. Testing and Reproducibility

Preprocessing is production code and should be tested.

Tests should cover:

- empty and missing input;
- already clean text;
- multilingual text;
- URLs and emails;
- HTML;
- repeated whitespace;
- punctuation;
- expected idempotence;
- malformed input.

In [ ]:
test_cases = [
    ("  Hello   world  ", "hello world"),
    ("Email A@B.COM", "email EMAIL_ADDRESS".lower()),
    ("<b>Useful</b>", "useful"),
    ("", ""),
    (None, ""),
]

for raw, expected in test_cases:
    actual = preprocess_text(raw)
    print(
        f"input={raw!r:<24} "
        f"actual={actual!r:<24} "
        f"passed={actual == expected}"
    )

## 16.1 Idempotence

A normalization function is **idempotent** when applying it twice produces the
same result as applying it once.

In [ ]:
idempotence_samples = [
    "  HELLO   world ",
    "<p>Email me at a@example.com</p>",
    "Visit https://example.com",
]

for sample in idempotence_samples:
    once = preprocess_text(sample)
    twice = preprocess_text(once)
    print(f"{sample!r:<40} idempotent={once == twice}")

## 16.2 Record the Configuration

Reproducible experiments should record:

- normalization code version;
- configuration values;
- language-specific rules;
- package versions;
- source-data version;
- examples used for testing.

In [ ]:
from dataclasses import asdict

pd.Series(
    asdict(default_config),
    name="Normalization configuration",
)

# 17. Evaluating Preprocessing Decisions

Text should not be judged as better merely because it looks cleaner.

Evaluate preprocessing using:

- retained information;
- vocabulary size;
- matching recall;
- model performance;
- robustness;
- error categories;
- performance by language and domain;
- reversibility and auditability.

In [ ]:
comparison_examples = pd.DataFrame(
    [
        ("I do not recommend it!!!", "i do not recommend it!!!"),
        ("Apple released a model.", "apple released a model."),
        ("The price is $250.", "the price is $250."),
        ("Great 😊", "great 😊"),
    ],
    columns=["Raw", "Conservatively normalized"],
)

comparison_examples["Information at risk"] = [
    "punctuation intensity",
    "entity capitalization",
    "none under this policy",
    "none under this policy",
]

comparison_examples

## 17.1 Downstream Vocabulary Illustration

Normalization can reduce redundant surface variation.

In [ ]:
vocabulary_documents = [
    "Model performance is GOOD",
    "model performance is good",
    "MODEL   performance is good",
]

raw_vocabulary = {
    token
    for document in vocabulary_documents
    for token in document.split()
}

normalized_vocabulary = {
    token
    for document in vocabulary_documents
    for token in preprocess_text(document).split()
}

print("Raw vocabulary:", sorted(raw_vocabulary))
print("Raw vocabulary size:", len(raw_vocabulary))
print()
print("Normalized vocabulary:", sorted(normalized_vocabulary))
print("Normalized vocabulary size:", len(normalized_vocabulary))

Reduced vocabulary can improve statistical efficiency, but only when the
merged variants are irrelevant to the task.

## 17.2 Common Preprocessing Errors

- deleting negation;
- removing entity capitalization;
- replacing meaningful numbers;
- deleting emojis from sentiment data;
- applying English rules to Arabic;
- changing valid names through spelling correction;
- stripping HTML with an unreliable rule;
- leaking information by fitting transformations before splitting;
- using different rules in training and production;
- overwriting raw data;
- failing to version the preprocessing configuration.

# 18. Knowledge Check

1. How do cleaning and normalization differ?
2. Why is preprocessing task-dependent?
3. Why should data be inspected before rules are written?
4. What is Unicode normalization?
5. When can whitespace normalization be harmful?
6. Why may placeholders be better than deletion?
7. How can lowercasing remove useful evidence?
8. Why should punctuation not always be removed?
9. Why are numbers not one uniform category?
10. How can emojis contribute meaning?
11. Why is stop-word removal risky?
12. Why should negation usually be preserved?
13. Which Arabic normalization decisions can lose information?
14. What does idempotence mean?
15. How should preprocessing quality be evaluated?

# 19. Exercises

## Exercise 1 — Data Inspection

Create a DataFrame containing HTML, URLs, emails, emojis, numbers, Arabic
text, missing values, and repeated whitespace. Produce an inspection report.

## Exercise 2 — Task-Specific Policy

Design separate preprocessing policies for sentiment analysis, Named Entity
Recognition, and financial extraction.

## Exercise 3 — Structured Placeholders

Extend the pipeline to mask dates, currency values, and record identifiers
with separate placeholders.

## Exercise 4 — Case Study

Compare lowercasing and case preservation on examples containing names,
acronyms, and ordinary words.

## Exercise 5 — Negation

Show how stop-word removal changes ten sentences containing `not`, `never`,
or `without`.

## Exercise 6 — Arabic Normalization

Apply several Arabic normalization policies to the same examples and document
which distinctions are lost.

## Exercise 7 — Unit Tests

Write at least fifteen automated tests for `preprocess_text`.

## Exercise 8 — Downstream Comparison

Train the same classifier with two preprocessing configurations. Compare macro
F1 and analyze the changed errors.

## Challenge Exercises

1. Implement language-aware preprocessing that selects English or Arabic
   normalization after validation.
2. Create a protected-term list so spelling correction cannot change product
   names and technical identifiers.
3. Build a preprocessing audit report showing every transformation applied to
   each record.
4. Compare conservative preprocessing with aggressive cleaning for a
   multilingual dataset.
5. Package the normalization pipeline as a tested Python module.

# 20. Summary and Next Lesson

In this lesson:

- preprocessing was separated into cleaning, normalization, tokenization, and
  later linguistic operations;
- task requirements determined which information should be preserved;
- Unicode normalization reduced representation inconsistencies;
- whitespace and HTML processing handled common raw-data artifacts;
- placeholders represented URLs, emails, mentions, and identifiers;
- case, punctuation, numbers, emojis, and repetition were treated as possible
  linguistic evidence rather than automatic noise;
- stop-word removal was shown to be optional and potentially harmful;
- Arabic normalization required explicit language-specific policies;
- a configurable function made preprocessing reproducible;
- unit tests and idempotence checks supported reliability;
- downstream evaluation determined whether a transformation was beneficial.

## Next Lesson

**Lesson 12: Tokenization and Sentence Segmentation** introduces word
tokenization, sentence-boundary detection, regular-expression tokenizers,
library tokenizers, multilingual tokenization, and tokenization evaluation.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Eisenstein, J. *Introduction to Natural Language Processing*.
- Python Unicode and regular-expression documentation.
- Unicode Standard Annexes.
- NLTK and spaCy text-processing documentation.
- Arabic NLP normalization and preprocessing literature.